In [2]:
import numpy as np
import pandas as pd


data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}


np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")


Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [3]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum


spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))


path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv"
df_transaksi = spark.read.csv(path_hdfs, header=True, inferSchema=True)

df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

print("Mesin Spark menyala dan data dari HDFS berhasil disiapkan!")

Mesin Spark menyala dan data dari HDFS berhasil disiapkan!


In [4]:

ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)


hasil_A = ringkasan_kota.join(df_target, on="kota", how="inner")
hasil_A = hasil_A.withColumn(
    "pencapaian_persen", 
    (col("total_pendapatan") / col("target_bulanan") * 100)
)


hasil_A = hasil_A.orderBy(col("pencapaian_persen").desc())
hasil_A.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [5]:
# B. Window Function - Kategori Terlaris per Kota


from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("pendapatan_kategori")
)


window_B = Window.partitionBy("kota").orderBy(col("pendapatan_kategori").desc())


hasil_B = pendapatan_kategori.withColumn("rank", row_number().over(window_B)) \
    .filter(col("rank") == 1) \
    .drop("rank")

hasil_B.show()

[Stage 11:>                                                         (0 + 1) / 1]

+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|  Magelang|Kesehatan & Kecan...|            7275000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|  Semarang|        Rumah Tangga|           11125000|
|      Solo|Kesehatan & Kecan...|            8425000|
|Yogyakarta|             Fashion|           13325000|
+----------+--------------------+-------------------+



In [6]:
# C. Spark SQL

df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")


hasil_C = spark.sql('''
    SELECT t.kota, tar.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tar ON t.kota = tar.kota
    GROUP BY t.kota, tar.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

hasil_C.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+

